# BSM L05E — Biometria jako *lokalna bramka* do sekretu (SecretBox + policy)

## Tryb pracy
Notebook służy do wypełnienia i wysłania odpowiedzi. Kod implementujesz lokalnie w **Android Studio / Kotlin** w starterze projektu do laboratorium.
## Link do projektu
Wklej link do repozytorium GitHub (to samo repo co w poprzednich labach):

- `TO_FILL_BEFORE_PUBLISH_WITH_GITHUB_REPO`


## Przygotowanie przed zajęciami (zrób zanim zaczniesz zadania)

### A. Import projektu w Android Studio (klik po kliku)
1. Uruchom Android Studio.
2. Wybierz: **File → Open...**
3. Wskaż folder projektu w sklonowanym repo, np. `student/apps/lesson_e_app`
4. Jeśli Android Studio zapyta o zaufanie do projektu: wybierz **Trust Project**.
5. Poczekaj na zakończenie synchronizacji Gradle (na dole zobaczysz pasek postępu).

### B. Jeśli Gradle Sync nie startuje lub stoi
1. Kliknij: **File → Sync Project with Gradle Files**
2. Jeśli pojawi się informacja o wymaganej wersji JDK, wybierz JDK 17:
   - **File → Settings → Build, Execution, Deployment → Build Tools → Gradle**
   - **Gradle JDK → 17**

### C. Uruchom testy jednostkowe (bez emulatora)
W tym labie wszystko sprawdzamy przez testy JVM.

Opcja (Android Studio):
1. Otwórz okno Gradle (zwykle po prawej).
2. Rozwiń: `lesson_e_app` → `app` → **Tasks** → **verification**
3. Uruchom: **testDebugUnitTest**

### D. Wygeneruj dowód do wysłania (jedna linia na zadanie)
Po uruchomieniu testów, w **Terminalu na swoim komputerze** (nie w Colab) wykonaj:
- `./gradlew :app:bsmEvidence`

To wypisze 3 linie zaczynające się od `E02|...`, `E03|...`, `E04|...`.
Te linie wklejasz do formularzy w tym notebooku.

Jeśli testy lub dowód nie działają, **nie idź dalej** — najpierw napraw środowisko.

## Cel labu
Wykład 5: biometria zwykle nie jest „sekretem do wysłania”, tylko mechanizmem lokalnym:
- biometria/credential **odblokowuje dostęp** do klucza / sekretu na urządzeniu,
- a dopiero potem aplikacja może wykonać wrażliwą operację.

W tym labie implementujesz:
1) `SecretBox` — minimalne szyfrowanie uwierzytelnione AES-GCM,
2) `BiometricBoundSecretStore` — politykę: sekret można ujawnić tylko z „świeżym” tokenem bramki.

Wszystko weryfikujesz przez testy w `app/src/test/...`.


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    wyslij_odpowiedz(task_id, final_answer)


# E02 — SecretBox: AES-GCM (szyfrowanie uwierzytelnione)

AES-GCM to tryb szyfrowania uwierzytelnionego (AEAD): jednocześnie zapewnia poufność i integralność.
W praktyce oznacza to, że nie tylko „nie da się odczytać” treści bez klucza, ale też nie da się jej sensownie podmienić bez wykrycia.
W tym zadaniu budujesz minimalny `SecretBox`, czyli małą warstwę API, która ukrywa detale `Cipher` i wymusza spójny format wiadomości.
Format `iv || ciphertextAndTag` jest celowo prosty: IV jest jawny i musi być przesyłany razem z szyfrogramem.
Kluczowa zasada: **nigdy nie używaj tego samego IV dwa razy z tym samym kluczem**.
W labie IV jest podawany jako parametr, żeby testy były deterministyczne, ale w realnej aplikacji IV generujesz losowo per wiadomość.
Wymagamy IV długości 12 bajtów, bo to standard dla GCM i zmniejsza ryzyko błędów konfiguracji.
Zwróć uwagę, że `decrypt(...)` ma zwrócić `null` przy błędzie autentykacji: to jest celowe, żeby nie wyciekały szczegóły przez wyjątki i komunikaty.

Plik: `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/SecretBox.kt`

## Dokładnie gdzie pisać kod
W `SecretBox.kt` uzupełnij:
- `TODO(L05-1)` w `encrypt(...)`
- `TODO(L05-2)` w `decrypt(...)`

## Wymagania (sprawdza je test)
- Algorytm: `AES/GCM/NoPadding`
- Tag: 128 bitów (już ustawione w `GCMParameterSpec`)
- IV ma mieć długość `SecretBox.IV_BYTES` (12 bajtów)
  - jeśli IV ma inną długość → rzuć `IllegalArgumentException`
- Zwracany format to **dokładnie**: `iv || ciphertextAndTag`
- `decrypt(...)` ma zwracać `null` jeśli:
  - wiadomość jest za krótka
  - autentykacja GCM nie przejdzie (ktoś zmienił bajty)

## Jak uruchomić testy tylko do tego zadania
Uruchom test:
- `app/src/test/java/com/example/secretlab/secure/SecretBoxStudentTest.kt`


In [ ]:
#@title E02 — Formularz odpowiedzi
secretbox_tests_passed = ""  #@param ["", "YES", "NO"]

# Wklej DOKŁADNIE linię z `./gradlew :app:bsmEvidence` zaczynającą się od `E02|`.
evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + secretbox_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E02", final_answer)


# E03 — BiometricBoundSecretStore: polityka „lokalnej bramki”

W tym zadaniu implementujesz ideę z wykładu: biometria (lub PIN/hasło urządzenia) jest **lokalną bramką**, a nie „sekretem do wysłania”.
Modelujemy to tokenem `GateToken`: jeśli użytkownik właśnie przeszedł weryfikację lokalną, aplikacja dostaje krótko ważny artefakt.
Twoim celem jest wymuszenie polityki czasu życia tokenu.
To ćwiczy typowy problem inżynierski: różne źródła czasu, drift, a także przypadek „token z przyszłości” (np. błąd zegara).
Wymaganie `0 <= age <= maxTokenAgeSeconds` jest celowo konserwatywne: jeśli nie potrafisz zaufać danym wejściowym, nie ujawniaj sekretu.
Z punktu widzenia bezpieczeństwa to zmniejsza ryzyko obejść bramki przez manipulację czasem albo błędy integracji.

Plik: `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/BiometricBoundSecretStore.kt`

## Dokładnie gdzie pisać kod
Uzupełnij `TODO(L05-3)` w metodzie:
- `revealSecret(token: GateToken?)`

## Polityka, którą masz wymusić
Ważne: `clock()` zwraca **epoch seconds** (sekundy), a nie milisekundy. `GateToken.issuedAtEpochSeconds` jest w tych samych jednostkach.

Sekret wolno ujawnić tylko wtedy, gdy:
1. `token` nie jest `null`
2. token jest wystarczająco „świeży”:
   - `age = clock() - token.issuedAtEpochSeconds`
   - wymaganie: `0 <= age <= maxTokenAgeSeconds`
   - jeśli `age < 0` (token „z przyszłości”) → traktuj jako niespełnione warunki i zwróć `null`

Jeśli warunki nie są spełnione → zwróć `null`.

## Jak uruchomić testy tylko do tego zadania
Uruchom test:
- `app/src/test/java/com/example/secretlab/secure/BiometricBoundSecretStoreStudentTest.kt`


In [ ]:
#@title E03 — Formularz odpowiedzi
store_tests_passed = ""  #@param ["", "YES", "NO"]

# Wklej DOKŁADNIE linię z `./gradlew :app:bsmEvidence` zaczynającą się od `E03|`.
evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + store_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E03", final_answer)


# E04 — Związanie szyfrogramu z kontekstem (AAD)

AAD (Additional Authenticated Data) pozwala uwierzytelnić dodatkowy kontekst bez szyfrowania go.
W praktyce to narzędzie do wiązania szyfrogramu z przeznaczeniem: np. „to jest seed TOTP dla tego labu”, a nie dowolny blob.
Jeśli ktoś skopiuje szyfrogram i spróbuje go użyć w innym miejscu (z innym kontekstem), odszyfrowanie powinno się nie udać.
To jest szczególnie ważne, gdy w aplikacji masz wiele typów sekretów przechowywanych w podobny sposób.
W tym labie kontekst jest stałą wartością bajtową, żeby polityka była jednoznaczna i łatwa do przetestowania.

Samo AES-GCM daje integralność, ale dopiero **AAD (Additional Authenticated Data)** pozwala związać szyfrogram z kontekstem użycia.
To praktycznie implementuje zasadę: „to zaszyfrowane dane do *konkretnego celu*, a nie dowolny blob do przenoszenia”.

1. `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/SecretBox.kt`
   - `TODO(L05-5)` w `encryptBound(...)`
   - `TODO(L05-6)` w `decryptBound(...)`

2. `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/BiometricBoundSecretStore.kt`
   - `TODO(L05-4)` w `setSecret(...)`

## Wymagania
- `encryptBound(...)` ma użyć AAD:
  - przed `doFinal(...)` wywołaj `cipher.updateAAD(context)`
- `decryptBound(...)` ma użyć tego samego `context` jako AAD
- jeśli ktoś spróbuje odszyfrować z innym kontekstem → zwróć `null`

## Test do uruchomienia
- `app/src/test/java/com/example/secretlab/secure/SecretBoxStudentTest.kt` (test `bindsCiphertextToContextWithAad`)


In [ ]:
#@title E04 — Formularz odpowiedzi
aad_tests_passed = ""  #@param ["", "YES", "NO"]

# Wklej DOKŁADNIE linię z `./gradlew :app:bsmEvidence` zaczynającą się od `E04|`.
evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + aad_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E04", final_answer)


# E05 — Checkpoint (uruchom cały pakiet testów)

Checkpoint sprawdza, czy cały pakiet testów przechodzi i czy dowód `bsmEvidence` ma komplet linii.
To jest etap, który minimalizuje nieporozumienia przy ocenianiu: wszyscy wysyłają ten sam format `evidence=...`.

Na koniec uruchom pełny pakiet testów (Android Studio):
- **Gradle → Tasks → verification → testDebugUnitTest**

Następnie wygeneruj dowód komendą lokalnie:
- `./gradlew :app:bsmEvidence`

Wpisz tylko, czy cały pakiet testów przeszedł.


In [ ]:
#@title E05 — Formularz odpowiedzi
all_tests_passed = ""  #@param ["", "YES", "NO"]

# Wklej DOKŁADNIE linię z `./gradlew :app:bsmEvidence` zaczynającą się od `E05|`.
evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + all_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E05", final_answer)
